## Example 1: Basic ExceptionGroup Creation and Handling

In [ ]:
# Example 1: Basic ExceptionGroup Creation and Handling

from exceptiongroup import ExceptionGroup

print("=== Basic ExceptionGroup Creation ===")

try:
    # Create individual exceptions
    exceptions = [
        ValueError("Invalid input value"),
        TypeError("Expected string, got int"),
        KeyError("Missing required key")
    ]
    
    # Create an ExceptionGroup
    error_group = ExceptionGroup("Multiple validation errors", exceptions)
    
    print(f"✓ Created ExceptionGroup: {error_group.message}")
    print(f"✓ Contains {len(error_group.exceptions)} exceptions:")
    
    for i, exc in enumerate(error_group.exceptions, 1):
        print(f"  {i}. {type(exc).__name__}: {exc}")
    
    # Test exception group handling
    try:
        raise error_group
    except ExceptionGroup as eg:
        print(f"\n✓ Caught ExceptionGroup with {len(eg.exceptions)} exceptions")
        print(f"✓ Message: {eg.message}")

except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"exceptiongroup example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed\n")

## Example 2: Nested ExceptionGroups and Hierarchical Error Handling

In [ ]:
# Example 2: Exception Group Splitting and Filtering

from exceptiongroup import ExceptionGroup

print("=== Exception Group Splitting ===")

try:
    # Create mixed exception types
    mixed_exceptions = [
        ValueError("Value error 1"),
        TypeError("Type error 1"), 
        ValueError("Value error 2"),
        RuntimeError("Runtime error 1"),
        TypeError("Type error 2")
    ]
    
    mixed_group = ExceptionGroup("Mixed exception types", mixed_exceptions)
    print(f"✓ Original group: {len(mixed_group.exceptions)} exceptions")
    
    # Split by exception type using built-in split method
    value_errors, remaining = mixed_group.split(ValueError)
    type_errors, runtime_errors = remaining.split(TypeError) if remaining else (None, remaining)
    
    print(f"✓ ValueError exceptions: {len(value_errors.exceptions) if value_errors else 0}")
    print(f"✓ TypeError exceptions: {len(type_errors.exceptions) if type_errors else 0}")
    print(f"✓ RuntimeError exceptions: {len(runtime_errors.exceptions) if runtime_errors else 0}")
    
    # Show split results
    if value_errors:
        for exc in value_errors.exceptions:
            print(f"  ValueError: {exc}")

except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"exceptiongroup example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed\n")

In [ ]:
# Example 3: Nested ExceptionGroups

from exceptiongroup import ExceptionGroup

print("=== Nested ExceptionGroups ===")

try:
    # Create nested exception structure
    user_errors = [ValueError("Invalid username"), TypeError("Age must be int")]
    profile_errors = [KeyError("Missing bio"), RuntimeError("Upload failed")]
    
    user_group = ExceptionGroup("User validation errors", user_errors)
    profile_group = ExceptionGroup("Profile validation errors", profile_errors)
    
    # Create top-level nested group
    all_errors = ExceptionGroup("Complete validation failed", [user_group, profile_group])
    
    print(f"✓ Created nested ExceptionGroup: {all_errors.message}")
    print(f"✓ Contains {len(all_errors.exceptions)} sub-groups")
    
    # Flatten nested exceptions
    def flatten_exceptions(eg):
        flattened = []
        for exc in eg.exceptions:
            if isinstance(exc, ExceptionGroup):
                flattened.extend(flatten_exceptions(exc))
            else:
                flattened.append(exc)
        return flattened
    
    flat_exceptions = flatten_exceptions(all_errors)
    print(f"✓ Total flattened exceptions: {len(flat_exceptions)}")
    
    for exc in flat_exceptions:
        print(f"  {type(exc).__name__}: {exc}")

except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"exceptiongroup example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 completed\n")

## Example 3: Concurrent Operations and Exception Aggregation

In [ ]:
# Example 4: Concurrent Error Aggregation

from exceptiongroup import ExceptionGroup

print("=== Concurrent Error Aggregation ===")

try:
    # A fixed failure plan, so the outcome is deterministic.
    failure_plan = {
        2: ValueError("Task 2: Invalid data"),
        4: RuntimeError("Task 4: Network timeout"),
        6: KeyError("Task 6: Missing config"),
    }

    def simulate_task(task_id):
        """Simulate a task that fails when the plan says so."""
        if task_id in failure_plan:
            raise failure_plan[task_id]
        return f"Task {task_id} completed successfully"
    
    # Simulate multiple tasks and collect exceptions
    results = []
    exceptions = []
    
    for task_id in range(1, 8):
        try:
            result = simulate_task(task_id)
            results.append(result)
            print(f"✓ Task {task_id}: Success")
        except (ValueError, RuntimeError, KeyError) as e:
            exceptions.append(e)
            print(f"✗ Task {task_id}: {type(e).__name__}")
    
    print(f"\n✓ Results: {len(results)} successful, {len(exceptions)} failed")
    assert len(results) == 4, results
    assert [type(e) for e in exceptions] == [ValueError, RuntimeError, KeyError], exceptions
    
    # Create exception group from failures
    failure_group = ExceptionGroup(f"Task execution failed", exceptions)
    print(f"✓ Created ExceptionGroup with {len(failure_group.exceptions)} exceptions")
    assert list(failure_group.exceptions) == exceptions
    value_errors, others = failure_group.split(ValueError)
    assert len(value_errors.exceptions) == 1 and len(others.exceptions) == 2, (value_errors, others)

except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"exceptiongroup example (cell 6) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 completed\n")

## Example 4: Exception Group Pattern Matching and Selective Handling

In [ ]:
# Example 5: Exception Group Pattern Matching

from exceptiongroup import ExceptionGroup

print("=== Exception Group Pattern Matching ===")

try:
    # Create exception group with different severities
    exceptions = [
        ValueError("Validation error"),
        MemoryError("Out of memory"),
        TypeError("Type mismatch"),
        OSError("File system error"),
        RuntimeError("Runtime issue")
    ]
    
    error_group = ExceptionGroup("System errors", exceptions)
    print(f"✓ Created group with {len(error_group.exceptions)} exceptions")
    
    # Pattern matching by severity
    def is_critical(exc):
        return isinstance(exc, (MemoryError, OSError))
    
    critical_errors, non_critical = error_group.split(is_critical)
    
    print(f"✓ Critical errors: {len(critical_errors.exceptions) if critical_errors else 0}")
    print(f"✓ Non-critical errors: {len(non_critical.exceptions) if non_critical else 0}")
    
    # Show critical errors
    if critical_errors:
        print("Critical errors found:")
        for exc in critical_errors.exceptions:
            print(f"  {type(exc).__name__}: {exc}")

except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"exceptiongroup example (cell 8) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 completed\n")

## Example 5: exceptiongroup functionality test 5

In [ ]:
# Example 6: Exception Group Utilities and Best Practices

from exceptiongroup import ExceptionGroup

print("=== Exception Group Utilities ===")

try:
    # Create comprehensive exception scenario
    validation_errors = [
        ValueError("Field 'name' is required"),
        ValueError("Invalid email format"),
        TypeError("Age must be numeric")
    ]
    
    system_errors = [
        ConnectionError("Database connection failed"),
        MemoryError("Insufficient memory"),
        RuntimeError("Service unavailable")
    ]
    
    # Create hierarchical structure
    validation_group = ExceptionGroup("Validation errors", validation_errors)
    system_group = ExceptionGroup("System errors", system_errors)
    master_group = ExceptionGroup("Application errors", [validation_group, system_group])
    
    print(f"✓ Master group: {master_group.message}")
    print(f"✓ Sub-groups: {len(master_group.exceptions)}")
    
    # Utility function for exception analysis
    def analyze_exception_group(eg):
        total = sum(len(sub.exceptions) if isinstance(sub, ExceptionGroup) 
                   else 1 for sub in eg.exceptions)
        return {"total_exceptions": total, "group_message": eg.message}
    
    analysis = analyze_exception_group(master_group)
    print(f"✓ Analysis: {analysis['total_exceptions']} total exceptions")
    print(f"✓ Message: {analysis['group_message']}")

except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"exceptiongroup example (cell 10) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 6 completed\n")